# Análisis exploratorio: Airbnb Bogotá

Este análisis consulta MongoDB (no lee los CSV directamente) y documenta estructura, calidad y distribuciones antes de transformar. Ejecuta las celdas en orden después de configurar `.env` según `README.md`.

**Fuente:** colecciones `Listings`, `Reviews` y `Calendar` de la base local MongoDB. Los resultados y conclusiones se calculan sobre los datos disponibles en esa base.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.extraccion import Extraccion

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
data = Extraccion().extraer()
list(data)

## 1. Entendimiento general y calidad

Se muestran muestras, dimensiones, tipos, resúmenes descriptivos, nulos y duplicados exactos por colección. Los nulos parciales se reportan para orientar transformaciones, no se eliminan automáticamente en bloque.

In [ ]:
quality = {}
for name, frame in data.items():
    print(f"\n{'=' * 80}\n{name}: primeras filas")
    display(frame.head())
    print(f"Dimensiones: {frame.shape[0]:,} filas x {frame.shape[1]:,} columnas")
    frame.info()
    print("Resumen de variables numéricas:")
    display(frame.describe(include="number").T)
    nulls = frame.isna().sum().sort_values(ascending=False)
    nulls = nulls[nulls > 0]
    duplicates = int(frame.duplicated().sum())
    quality[name] = {"rows": len(frame), "columns": len(frame.columns),
                     "duplicates": duplicates, "nulls": nulls}
    print(f"Duplicados exactos: {duplicates:,}")
    print("Nulos por columna (solo columnas con nulos):")
    display(nulls.to_frame("missing_count"))
    print("Variables relevantes disponibles:",
          [c for c in frame.columns if c in {
              "price", "minimum_nights", "availability_365", "date",
              "review_scores_rating", "neighbourhood_cleansed", "room_type"
          }])

## 2. Distribuciones y valores atípicos

Los histogramas y diagramas de caja usan variables numéricas presentes en cada colección. Los límites del rango intercuartílico (IQR) se reportan como señal de revisión: un valor fuera de esos límites no implica por sí solo un error.

In [ ]:
variables = ["price", "minimum_nights", "availability_365"]
outlier_summary = {}
for name, frame in data.items():
    present = [c for c in variables if c in frame.columns]
    if not present:
        print(f"{name}: no contiene las variables objetivo para estos gráficos.")
        continue
    fig, axes = plt.subplots(len(present), 2, figsize=(13, 4 * len(present)))
    if len(present) == 1:
        axes = [axes]
    for column, (hist_ax, box_ax) in zip(present, axes):
        values = pd.to_numeric(frame[column], errors="coerce").dropna()
        sns.histplot(values, kde=len(values) > 1, ax=hist_ax)
        hist_ax.set_title(f"{name}: distribución de {column}")
        sns.boxplot(x=values, ax=box_ax)
        box_ax.set_title(f"{name}: caja de {column}")
        if len(values):
            q1, q3 = values.quantile([0.25, 0.75])
            iqr = q3 - q1
            count = int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum())
            outlier_summary[(name, column)] = {
                "n": len(values), "outliers_iqr": count,
                "min": values.min(), "median": values.median(), "max": values.max()
            }
    plt.tight_layout()
    plt.show()
pd.DataFrame.from_dict(outlier_summary, orient="index").rename_axis(
    ["collection", "variable"]
)

## 3. Campos anidados, fechas y relaciones

El bloque identifica columnas con estructura compleja, revisa relaciones numéricas cuando hay datos suficientes y resume el calendario por mes si contiene `date`. La correlación describe asociación lineal y no causalidad.

In [ ]:
for name, frame in data.items():
    nested = [c for c in frame.columns if frame[c].map(
        lambda value: isinstance(value, (dict, list, tuple))
    ).any()]
    print(f"{name}: columnas con objetos/listas anidadas: {nested or 'ninguna detectada'}")
    numeric = frame.select_dtypes(include="number")
    if numeric.shape[1] >= 2:
        corr = numeric.corr(min_periods=2)
        plt.figure(figsize=(min(14, 1 + corr.shape[1]), min(10, 1 + corr.shape[1])))
        sns.heatmap(corr, cmap="vlag", center=0, annot=False)
        plt.title(f"Correlación numérica: {name}")
        plt.tight_layout()
        plt.show()
        pairs = corr.where(~pd.DataFrame(
            [[i == j for j in corr.columns] for i in corr.index],
            index=corr.index, columns=corr.columns
        )).stack().sort_values(key=abs, ascending=False)
        print("Pares con mayor correlación absoluta:")
        display(pairs.head(10).to_frame("correlation"))

calendar = data.get("Calendar", pd.DataFrame())
if "date" in calendar.columns:
    calendar_dates = pd.to_datetime(calendar["date"], errors="coerce", format="mixed")
    monthly = calendar.assign(month=calendar_dates.dt.to_period("M")).dropna(subset=["month"])
    monthly_counts = monthly.groupby("month").size().rename("records")
    display(monthly_counts.to_frame())
    monthly_counts.plot(figsize=(12, 4), marker="o", title="Registros de Calendar por mes")
    plt.ylabel("Registros")
    plt.tight_layout()
    plt.show()
else:
    print("Calendar no contiene una columna date; no es posible agrupar por mes.")

## 4. Hallazgos y decisiones para la transformación

El resumen siguiente se calcula al ejecutar el Notebook. Debe revisarse e interpretarse antes de incorporarlo al informe final; la detección estadística no reemplaza la validación del dominio.

In [ ]:
for name, metrics in quality.items():
    null_columns = metrics["nulls"].index.tolist()
    null_total = int(metrics["nulls"].sum())
    display(Markdown(
        f"### {name}\n"
        f"- Registros y columnas observados: **{metrics['rows']:,} x {metrics['columns']:,}**.\n"
        f"- Duplicados exactos: **{metrics['duplicates']:,}**. Revisar y eliminar "
        f"duplicados exactos en la transformación, si se confirma que no representan eventos repetidos.\n"
        f"- Nulos registrados: **{null_total:,}** en {len(null_columns)} columnas. "
        f"Conservar nulos parciales evita descartar filas completas; decidir imputaciones según el uso analítico.\n"
        f"- Columnas con nulos: {', '.join(null_columns[:15]) if null_columns else 'ninguna'}."
    ))

display(Markdown(
    "### Decisiones propuestas para ETL\n"
    "1. Eliminar filas vacías y duplicados exactos, y conservar nulos parciales para "
    "no sesgar el tamaño de las colecciones.\n"
    "2. Normalizar `price` quitando símbolos monetarios/separadores y convertirlo a numérico; "
    "validar los valores que no puedan convertirse.\n"
    "3. Estandarizar fechas y derivar año, mes, día y trimestre desde `date`.\n"
    "4. Aplanar documentos anidados y mantener listas serializadas para conservar el detalle; "
    "derivar el conteo de amenities cuando aplique.\n"
    "5. Revisar valores atípicos con estadísticas y contexto de negocio antes de excluirlos; "
    "la implementación no elimina outliers automáticamente."
))